# ENTSO-E Feature Engineering

Features are built for a forecast made around 11:00 on D-1, for delivery day D. `create_features` builds every series once.

The TFT feature set grows in three steps. D1 is the Polish market and macro factors. D2 adds Polish market coupling (JAO). D3 adds foreign markets and systems. ENTSO-E supplies part of D1 and part of D3, so two files are saved: `d1` and `d3`. Each file holds only the columns its step adds. `select_features(df, model, step)` keeps one of them: `model` is `"tft"` and `step` is `"d1"` or `"d3"`.

**Known and unknown columns.** Unknown columns are not shifted to hide day D. They are passed to the `TimeSeriesDataSet` as unknown reals, so the model only sees them up to D-1.

* The Polish day-ahead price of day D is the target and is not a feature. `PL_price_lag24`, `PL_price_lag48` and `PL_price_lag168` use only earlier days.
* The day-ahead `load_forecast` for day D is known. `PL_load_fc_delta24` is that forecast minus the forecast for the same hour of D-1.
* Wind, solar and total generation forecasts (`PL_wind_fc`, `PL_solar_fc`, `PL_gen_fc`) are not shifted. They are unknown.
* A spread is the Polish price minus the foreign price in the same hour, not shifted. It is unknown. Names shorten `DE_LU` to `DE` and `SE_4` to `SE4`, except `spread_PL_DE_LU`, which keeps the bidding-zone code.
* Actual coal generation is shifted by 24 hours (`PL_coal_gen_lag24`) and is unknown. Coal is hard coal, lignite and coal-derived gas, summed.
* Foreign day-ahead load forecasts are known.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.file_io import (
    load_entsoe_zones,
    save_merged_data,
    inspect_merged_file,
)

from src.utils.feature_engineering.fe_entsoe import (
    COLUMNS,
    create_features,
    select_features,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
data_entsoe = load_entsoe_zones(DATA_DIR / "preprocessed" / "entsoe" / date_range)

## 1. Input Columns

Preprocessed series per bidding zone, before any shift or aggregation.

In [3]:
for country, data in data_entsoe.items():
    print(f"=== {country} ===")
    print(f"{list(data.columns)}")

=== CZ ===
['load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast', 'generation_forecast_Scheduled_Consumption', 'generation_forecast_Scheduled_Aggregated']
=== DE_LU ===
['load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast_Solar', 'generation_wind_solar_forecast_Wind_Offshore', 'generation_wind_solar_forecast_Wind_Onshore', 'generation_forecast']
=== LT ===
['load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast_Solar', 'generation_wind_solar_forecast_Wind_Onshore']
=== PL ===
['load', 'generation_Biomass', 'generation_Fossil_Brown_coal_Lignite', 'generation_Fossil_Coal_derived_gas', 'generation_Fossil_Gas', 'generation_Fossil_Hard_coal', 'generation_Fossil_Oil', 'generation_Hydro_Pumped_Storage', 'generation_Hydro_Run_of_river_and_pondage', 'generation_Hydro_Water_Reservoir', 'generation_Solar', 'generation_Wind_Onshore', 'load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast_Solar', 'generation_wind_solar_forecast_Wind_Onsh

## 2. Features

`create_features` builds the 19 columns of the TFT feature set.

**Poland.** `PL_price_lag24`, `PL_price_lag48`, `PL_price_lag168`. `PL_load_fc`, `PL_load_fc_delta24`, `PL_wind_fc`, `PL_solar_fc`, `PL_gen_fc` (total generation forecast), `PL_coal_gen_lag24`.

**Foreign.** Spreads versus DE_LU, CZ, SK, SE4 and LT. Day-ahead load for DE, CZ, SK, SE4 and LT.

In [4]:
features = create_features(data_entsoe)

In [5]:
print(features.shape)
print(list(features.columns))

(38281, 19)
['PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc', 'PL_solar_fc', 'PL_gen_fc', 'PL_coal_gen_lag24', 'spread_PL_DE_LU', 'spread_PL_CZ', 'spread_PL_SK', 'spread_PL_SE4', 'spread_PL_LT', 'DE_load_fc', 'CZ_load_fc', 'SK_load_fc', 'SE4_load_fc', 'LT_load_fc']


## 3. The TFT Steps

`COLUMNS["tft"]` lists the ENTSO-E part of each step.

**D1 (9 columns).** Price history: `PL_price_lag24`, `PL_price_lag48`, `PL_price_lag168`. Polish demand and generation: `PL_load_fc`, `PL_load_fc_delta24`, `PL_wind_fc`, `PL_solar_fc`, `PL_gen_fc`, `PL_coal_gen_lag24`.

**D3 (10 columns).** Five price spreads: `spread_PL_DE_LU`, `spread_PL_CZ`, `spread_PL_SK`, `spread_PL_SE4`, `spread_PL_LT`. Five foreign load forecasts: `DE_load_fc`, `CZ_load_fc`, `SK_load_fc`, `SE4_load_fc`, `LT_load_fc`.

D2 has no ENTSO-E column.

In [6]:
for model, steps in COLUMNS.items():
    for step in steps:
        subset = select_features(features, model, step)
        print(f"=== {model}/{step} ===")
        print(subset.shape[1], "features")
        print(subset.index.min())
        print(subset.index.max())
        print(list(subset.columns))

=== tft/d1 ===
9 features
2022-06-08 22:00:00+00:00
2026-10-20 22:00:00+00:00
['PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc', 'PL_solar_fc', 'PL_gen_fc', 'PL_coal_gen_lag24']
=== tft/d3 ===
10 features
2022-06-08 22:00:00+00:00
2026-10-20 22:00:00+00:00
['spread_PL_DE_LU', 'spread_PL_CZ', 'spread_PL_SK', 'spread_PL_SE4', 'spread_PL_LT', 'DE_load_fc', 'CZ_load_fc', 'SK_load_fc', 'SE4_load_fc', 'LT_load_fc']


## 4. Conclusions

D1 and D3 are saved for TFT, two files in total. Files go to `data/features/entsoe/tft/{d1|d3}/`, and each file is verified by reading it back.

The files cover 2022-06-08 22:00 to 2026-10-20 22:00 UTC. The only missing values are in the first 7 days, which are needed to build the lagged features (the first complete hour is 2022-06-16 00:00 local time), and in the last hour, where only the day-ahead price exists.

## 5. Saving Feature Data

In [7]:
for model, steps in COLUMNS.items():
    for step in steps:
        path = DATA_DIR / "features" / "entsoe" / model / step / f"{date_range}.parquet"
        save_merged_data(select_features(features, model, step), path)
        inspect_merged_file(path)

  [FILE] data/features/entsoe/tft/d1/2022-06-09_2026-10-21.parquet | shape: (38281, 9) | idx: 2022-06-08 22:00:00+00:00-2026-10-20 22:00:00+00:00 | cols: 9 | NaN: 293
  [FILE] data/features/entsoe/tft/d3/2022-06-09_2026-10-21.parquet | shape: (38281, 10) | idx: 2022-06-08 22:00:00+00:00-2026-10-20 22:00:00+00:00 | cols: 10 | NaN: 5
